In [ ]:
import requests
import os

# 1. アナロジーデータをダウンロード
url = "http://download.tensorflow.org/data/questions-words.txt"
analogy_file = "questions-words.txt"
if not os.path.exists(analogy_file):
    r = requests.get(url)
    with open(analogy_file, "wb") as f:
        f.write(r.content)

# 2. データを読み込んで capital-common-countries セクションだけを抽出
analogy_data = []
capture = False

with open(analogy_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line.startswith(":"):
            capture = line == ": capital-common-countries"
            continue
        if capture:
            words = line.split()
            if len(words) == 4:
                analogy_data.append(words)

print(f"✅ capital-common-countries セクションの問題数: {len(analogy_data)}")

# 3. 各アナロジーに対してベクトル演算と最類似単語を記録
results = []

for w1, w2, w3, actual in analogy_data:
    try:
        predicted, score = model.most_similar(positive=[w2, w3], negative=[w1], topn=1)[0]
        results.append((w1, w2, w3, actual, predicted, score))
    except KeyError:
        results.append((w1, w2, w3, actual, "N/A", 0.0))

# 4. 最初の10件だけ表示
print("🧠 例（先頭10件）:")
for row in results[:10]:
    print(f"{row[0]:<10} {row[1]:<10} {row[2]:<10} → 正解: {row[3]:<10} / 予測: {row[4]:<10} / 類似度: {row[5]:.4f}")

# 5. CSVで保存したい場合（オプション）
import csv

with open("capital_analogy_results.csv", "w", newline='', encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["w1", "w2", "w3", "actual", "predicted", "similarity"])
    writer.writerows(results)

print("✅ 結果を 'capital_analogy_results.csv' に保存しました。")
